# Setup

In [1]:
from dataclasses import dataclass

@dataclass
class TrainConfig:
    random_state = 42

    # Bermant's SSL
    ssl_config_path = (
        "configs/pipeline_watkins.json"
    )
    embedding_model_path = (
        "watkins/models/final_model_v1.pt"
        # "dominica/models/final_model_v1.pt"
        # "both/models/final_model_v1.pt"
    )

    # DATASET PARAMS
    hop_len = 48
    sample_rate = 48000
    window = 0.5
    window_pad = 136
    epsilon = 2e-6
    watkins_wavs_path = """../../data/wavs48khz_watkins/*.wav"""
    watkins_selections_path = """../../data/selections/*.txt"""
    dominica_wavs_path = """../../data/Dominica_dataset/Signal_parts/*.wav"""
    dominica_annotation_path = """../../data/Dominica_dataset/Annotations_Dominica.mat"""

    # Model Params
    lstm_input_size=32
    lstm_hidden_size=128
    lstm_num_layers=5
    epochs = 250

In [2]:
import numpy as np
import torch
import random

# Manual Seeding
random.seed(TrainConfig.random_state)
np.random.seed(TrainConfig.random_state)
torch.manual_seed(TrainConfig.random_state)
torch.cuda.manual_seed_all(TrainConfig.random_state)

In [3]:
# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
import warnings

# Suppress the specific FutureWarning from librosa
warnings.filterwarnings("ignore", category=FutureWarning, module="librosa")
warnings.filterwarnings(
    "ignore", message="PySoundFile failed. Trying audioread instead."
)

In [5]:
import json
import torch.nn as nn
from ssl_model.models import SpectralBoundaryEncoder

def load_embedding_model(path: str, model_params: dict) -> torch.nn.Module:
    model = SpectralBoundaryEncoder(**model_params)
    model.load_state_dict(torch.load(path, map_location=device, weights_only=False))
    model.eval().to(device)
    return model


with open(TrainConfig.ssl_config_path, "r") as f:
    SSL_CONFIG = json.load(f)

embedding_model = load_embedding_model(TrainConfig.embedding_model_path, SSL_CONFIG["model"])

# Data Preparation

## Watkins Dataset

In [6]:
import os
import pandas as pd
from torch.utils.data import Dataset
import librosa
import soundfile as sf
from pathlib import Path


def _get_files(pattern: str) -> list[Path]:
    """Resolve file pattern to sorted list of Path objects."""
    path = Path(pattern)
    return sorted(path.parent.glob(path.name))


class WatkinsClicks(Dataset):
    def __init__(
            self,
            wavs_path: str = TrainConfig.watkins_wavs_path,
            annotation_path: str = TrainConfig.watkins_selections_path,
            augment: bool = False) -> None:
        self.window = TrainConfig.window
        self.window_pad = TrainConfig.window_pad
        self.sample_rate = TrainConfig.sample_rate
        self.seed = TrainConfig.random_state
        self.epsilon = TrainConfig.epsilon
        self.hop_len = TrainConfig.hop_len
        self.augment = augment

        # Initialize randomness
        self._set_seeds()

        # Data loading
        self.raw_annotations = self._load_annotations(annotation_path, wavs_path)
        self.files = _get_files(wavs_path)

        # Sample precomputation
        self.positive_samples = self._precompute_samples()

    def _set_seeds(self) -> None:
        """Initialize all relevant random seeds."""
        np.random.seed(self.seed)
        random.seed(self.seed)
        torch.manual_seed(self.seed)

    def _load_annotations(self, path: str, base_path: str) -> dict[str, list[float]]:
        # Load annotations
        selections = _get_files(path)
        annotations = {}
        for path in selections:
            name = Path(path).name.split(".")[0]
            df = pd.read_csv(path, delimiter="\t")
            click_times = df["Begin Time (s)"].tolist()[::2]
            annotations[name] = list(click_times)
        return annotations

    def _precompute_samples(
        self,
    ) -> tuple[list[tuple[Path, float]], list[tuple[Path, float]]]:
        positive_samples = []
        annotations = {}
        for wav_path in self.files:
            wav_path = wav_path.as_posix()
            dur = sf.info(wav_path).duration
            file_name = os.path.splitext(os.path.basename(wav_path))[0]
            if file_name in self.raw_annotations:
                a = [0] * int(dur * self.sample_rate // self.hop_len + 1)
                for click_time in self.raw_annotations[file_name]:
                    # Adjust start time so that click is inside the window
                    start_time = max(0.0, click_time - self.epsilon)
                    a[int(start_time * self.sample_rate // self.hop_len)] = 1
                    # Ensure window fits within file duration
                    if dur < self.window:
                        continue
                    max_start = dur - (self.window + self.epsilon)
                    if start_time > max_start:
                        start_time = max_start
                    positive_samples.append((wav_path, start_time, dur))
                annotations[wav_path] = a
        self.annotations = annotations
        return positive_samples

    def __len__(self) -> int:
        return len(self.positive_samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        wav, start_time, dur = self.positive_samples[idx]

        # Introduce a small random offset to the start_time
        # Add random offset if augmentation is enabled
        if self.augment:
            max_offset = self.window - self.epsilon  # Maximum offset in seconds
            offset = random.uniform(-max_offset, max_offset)
            start_time = start_time + offset
            # Ensure we stay within valid bounds
            start_time = max(0.0, min(start_time, dur - self.window - self.epsilon))

        x, _ = librosa.load(
            wav,
            offset=start_time,
            duration=self.window + self.epsilon,
            sr=self.sample_rate,
        )

        # Compute total number of frames (samples).
        window_frames = int(self.window * self.sample_rate) + self.window_pad
        x = librosa.util.fix_length(data=x, size=window_frames)
        # Convert to tensor with a channel dimension.
        x = torch.tensor(x, dtype=torch.float32).unsqueeze(dim=0)
        # At this point, x has shape [1, window_frames].

        labels = self.annotations[wav][int(start_time * self.sample_rate // self.hop_len):int((start_time + self.window) * self.sample_rate // self.hop_len + 1)]
        return x, torch.tensor(labels, dtype=torch.float32)

## Dominica Dataset

In [7]:
import scipy.io as sio

class DominicaClicks(WatkinsClicks):
    def __init__(
            self,
            wavs_path: str = TrainConfig.dominica_wavs_path,
            annotation_path: str = TrainConfig.dominica_annotation_path,
            augment: bool = False) -> None:
        super().__init__(wavs_path, annotation_path, augment)

    def _load_annotations(self, path: str, base_path: str) -> dict[str, list[float]]:
        """Load and parse MATLAB annotations file."""
        mat_data = sio.loadmat(path)["Annotations_Dominica"]
        return {
            f"{entry[0][0].item()[:-4]}": entry[1][0].tolist() if entry[1].size > 0 else []
            for entry in mat_data[1:]  # Skip header
        }

## Train/Test Dataset Creation

In [8]:
from torch.utils.data import DataLoader, ConcatDataset, random_split

BATCH_SIZE = 32
train_size = 0.85
val_size = 1 - train_size

selection_dataset = WatkinsClicks(
    augment=True,
)
train_selection_dataset, val_selection_dataset = random_split(selection_dataset, [train_size, val_size], torch.Generator().manual_seed(42))

dominica_dataset = DominicaClicks(
    augment=True,
)
train_dominica_dataset, val_dominica_dataset = random_split(dominica_dataset, [train_size, val_size], torch.Generator().manual_seed(42))

train_dataset = ConcatDataset([
    train_selection_dataset,
    # train_dominica_dataset,
])
val_dataset = ConcatDataset([
    val_selection_dataset,
    # val_dominica_dataset,
])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [9]:
len(train_selection_dataset), len(train_dominica_dataset), len(train_dataset)

(1596, 24055, 1596)

In [10]:
len(val_selection_dataset), len(val_dominica_dataset), len(val_dataset)

(281, 4245, 281)

# Model Prep

## Loss

In [11]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = torch.nn.functional.binary_cross_entropy_with_logits(
            inputs, targets, reduction="none"
        )
        # p if target = 1, 1-p if target = 0
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss

## Metrics

In [12]:
import torch
from torchmetrics.classification import BinaryPrecision, BinaryRecall, BinaryF1Score
from torchmetrics import MetricCollection
from sklearn.metrics import precision_recall_curve, auc, f1_score


class MetricManager:
    def __init__(
        self,
        device: str = "cuda" if torch.cuda.is_available() else "cpu",
        threshold: float = 0.45,
        selected_metrics: list = None,
        dynamic_threshold: bool = False,
    ):
        """
        Args:
            device (str): Device to run the metrics on.
            threshold (float): Initial threshold for binary metrics.
            selected_metrics (list): List of metric names to compute.
                Options can include: "precision", "recall", "f1", "pr_auc", "r_value".
                If None, defaults to ["precision", "recall", "f1"].
            dynamic_threshold (bool): If True, dynamically search for the threshold (nearest 5 candidates with step 0.01)
                                      that maximizes F1 score, using the current epoch's predictions.
        """
        self.device = device
        self.threshold = threshold
        self.dynamic_threshold = dynamic_threshold

        if selected_metrics is None:
            selected_metrics = ["precision", "recall", "f1"]
        self.selected_metrics = selected_metrics

        metric_dict = {}
        if "precision" in selected_metrics:
            metric_dict["precision"] = BinaryPrecision(threshold=threshold)
        if "recall" in selected_metrics:
            metric_dict["recall"] = BinaryRecall(threshold=threshold)
        if "f1" in selected_metrics:
            metric_dict["f1"] = BinaryF1Score(threshold=threshold)
        # Additional torchmetrics can be added here if needed.

        self.metrics = MetricCollection(metric_dict).to(self.device)

        # These two are computed externally (if selected).
        self.compute_pr_auc = "pr_auc" in self.selected_metrics
        self.compute_r_value = "r_value" in self.selected_metrics

        # For PR-AUC and threshold optimization, accumulate predictions and targets.
        self.all_preds = []
        self.all_targets = []

        # Dummy buffer to simulate nn.Module behavior for device placement.
        self.register_buffer("dummy", torch.tensor(0))

    def update(self, preds: torch.Tensor, targets: torch.Tensor):
        """Update internal metrics and accumulate predictions for PR-AUC and threshold optimization."""
        self.metrics.update(preds, targets)
        self.all_preds.append(preds.detach().cpu())
        self.all_targets.append(targets.detach().cpu())

    def optimize_threshold(self):
        """
        Searches for the best threshold within a small neighborhood of the current threshold.
        It checks the nearest 5 thresholds with a step of 0.01 (i.e. a window of ±0.02 around the current threshold),
        and updates the threshold based on the F1 score.
        """
        current_thresh = self.threshold
        lower = max(0.0, current_thresh - 0.02)
        upper = min(1.0, current_thresh + 0.02)
        candidate_thresholds = np.linspace(lower, upper, 5)

        combined_preds = torch.cat(self.all_preds).numpy()
        combined_targets = torch.cat(self.all_targets).numpy()

        best_thresh = current_thresh
        best_f1 = 0.0
        for thresh in candidate_thresholds:
            preds_bin = (combined_preds >= thresh).astype(np.float32)
            current_f1 = f1_score(combined_targets, preds_bin, zero_division=0)
            if current_f1 > best_f1:
                best_f1 = current_f1
                best_thresh = thresh

        # Update threshold in this manager.
        self.threshold = best_thresh
        # Update threshold for all torchmetrics that support it.
        for metric in self.metrics.values():
            if hasattr(metric, "threshold"):
                metric.threshold = best_thresh

        return best_thresh, best_f1

    def compute(self) -> dict:
        """Compute and return selected metrics.
        Optionally, dynamically update the threshold before computing.
        Returns only the metrics that are selected.
        """

        # Optionally update the threshold dynamically
        if self.dynamic_threshold and len(self.all_preds) > 0:
            self.optimize_threshold()

        metrics = self.metrics.compute()
        output = {k: v.item() if hasattr(v, "item") else v for k, v in metrics.items()}

        # Compute PR-AUC if requested.
        if self.compute_pr_auc:
            try:
                concatenated_targets = torch.cat(self.all_targets).numpy()
                concatenated_preds = torch.cat(self.all_preds).numpy()
                precisions, recalls, _ = precision_recall_curve(
                    concatenated_targets, concatenated_preds
                )
                output["pr_auc"] = auc(recalls, precisions)
            except ValueError:
                output["pr_auc"] = float("nan")

        # Compute r_value if requested.
        if self.compute_r_value:
            combined_preds = torch.cat(self.all_preds)
            combined_targets = torch.cat(self.all_targets)
            num_positives = int(combined_targets.sum().item())
            if num_positives > 0:
                _, top_indices = torch.topk(combined_preds, k=num_positives)
                output["r_value"] = combined_targets[top_indices].float().mean().item()
            else:
                output["r_value"] = 0.0

        # Clear accumulated predictions and targets for the next epoch.
        # self.all_preds.clear()
        # self.all_targets.clear()

        return output

    def reset(self):
        """Reset all internal metrics and accumulated predictions/targets."""
        self.metrics.reset()
        self.all_preds.clear()
        self.all_targets.clear()

    def register_buffer(self, name: str, tensor: torch.Tensor):
        """Simulate nn.Module.register_buffer for proper device placement."""
        setattr(self, name, tensor)

## Model

In [13]:
import torch
import torch.nn as nn
import pytorch_lightning as pl


class LitEmbeddingLSTMModel(pl.LightningModule):
    def __init__(
            self,
            embedding_model,
            lstm_input_size=TrainConfig.lstm_input_size,
            lstm_hidden_size=TrainConfig.lstm_hidden_size,
            lstm_num_layers=TrainConfig.lstm_num_layers,
            learning_rate=1e-3,
    ):
        super().__init__()
        self.example_input_array = torch.randn(1, 1, 24136)
        self.learning_rate = learning_rate

        # Example: assume self.embedding_model and some new layers are defined
        self.embedding_model = embedding_model  # Pre-trained
        # Freeze all parameters of the embedding model
        for name, param in self.embedding_model.named_parameters():
            param.requires_grad = False
        # Unfreeze the last N parameters
        self.embedding_model.train()
        unfreeze_count = 10
        params = list(self.embedding_model.parameters())
        for param in params[-unfreeze_count:]:
            param.requires_grad = True

        # New layers (replace with your actual model)
        self.lstm = nn.LSTM(
            input_size=lstm_input_size,
            hidden_size=lstm_hidden_size,
            num_layers=lstm_num_layers,
            
            batch_first=True,
            bidirectional=True,
            dropout=0.3,
        )
        self.fc = nn.Linear(2 * lstm_hidden_size, 1)

        # Loss function (FocalLoss, as in the CNN version)
        self.loss_fn = FocalLoss(alpha=0.3, gamma=3.5, reduction="mean")

        # Metrics
        self.threshold = 0.5
        self.val_metrics = MetricManager(
            threshold=self.threshold, dynamic_threshold=False
        )
        # self.train_metrics = MetricManager(
        #     threshold=self.threshold, dynamic_threshold=False
        # )
        self.best_val_f1 = 0.0

    # def forward(self, x):
    #     # Get embeddings from the embedding model (shape [B, T, 32])
    #     embedding = self.embedding_model(x)
    #     # Pass embeddings through LSTM and FC layers (example)
    #     lstm_out, _ = self.lstm(embedding)
    #     logits = self.fc(lstm_out)
    #     logits = logits.squeeze(-1)  # [B, T]
    #     return logits
    def forward(self, x, return_sequence=False):
        """
        If return_sequence is True, returns the full [B, T, H] LSTM outputs,
        otherwise returns [B, T] logits as before.
        """
        # 1) Base embeddings
        # print(x.shape)
        emb = self.embedding_model(x)                  # [B, T, 32]
        # print(emb.shape)
        # 2) LSTM
        lstm_out, _ = self.lstm(emb)                  # [B, T, 2*hidden_size]
        if return_sequence:
            return lstm_out
        # 3) Final logits
        logits = self.fc(lstm_out).squeeze(-1)        # [B, T]
        return logits

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        x, labels = batch
        # Grab raw LSTM features
        seq_feats = self(x, return_sequence=True)     # [B, T, F]
        return {"feats": seq_feats.cpu(), "labels": labels.cpu()}

    def training_step(self, batch, batch_idx):
        inputs, targets = batch  # (inputs, targets)
        logits = self(inputs)
        loss = self.loss_fn(logits, targets)

        # predictions = torch.sigmoid(logits)
        # self.train_metrics.update(predictions.flatten(), targets.flatten())

        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, targets = batch
        logits = self(inputs)
        loss = self.loss_fn(logits, targets)
        predictions = torch.sigmoid(logits)

        # Update metrics
        self.val_metrics.update(predictions.flatten(), targets.flatten())

        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def on_validation_epoch_end(self):
        # Compute all metrics
        metrics = self.val_metrics.compute()
        for name, value in metrics.items():
            self.log(f"val_{name}", value, prog_bar=f"val_{name}")
        # Reset metrics
        self.val_metrics.reset()

    def configure_optimizers(self):
        # Separate parameters into two groups: embedding and new layers.
        embedding_params = []
        new_params = []
        for name, param in self.named_parameters():
            if "embedding_model" in name:
                embedding_params.append(param)
            else:
                new_params.append(param)

        optimizer = torch.optim.AdamW(
            [
                {"params": embedding_params, "lr": self.learning_rate * 0.1},
                {"params": new_params, "lr": self.learning_rate},
            ],
            weight_decay=1e-4,
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=5, eta_min=1e-7
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
                "frequency": 1,
            },
        }

    def configure_gradient_clipping(self, optimizer, gradient_clip_val, gradient_clip_algorithm):
        # Clip gradients to improve stability, if desired.
        torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=1.0)

# Training

In [14]:
# Instantiate the Lightning model
model = LitEmbeddingLSTMModel(embedding_model).to(device)

In [15]:
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint

# Initialize logger
tb_logger = TensorBoardLogger(
    save_dir="lstm-detector/", name="lstm_model", log_graph=True  # Log model computational graph
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_f1",
    mode="max",
    save_top_k=1,
    filename="best-lstm-detector",
    save_weights_only=False,
    verbose=False,
)

# Initialize Trainer
trainer = pl.Trainer(
    logger=tb_logger,
    max_epochs=TrainConfig.epochs,
    accelerator=device,
    devices=1,
    check_val_every_n_epoch=5,
    callbacks=[checkpoint_callback],
    enable_checkpointing=True,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [ ]:
from tqdm.notebook import tqdm
# Start training
trainer.fit(model, train_loader, val_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name            | Type                    | Params | Mode  | In sizes      | Out sizes                                    
------------------------------------------------------------------------------------------------------------------------------------
0 | embedding_model | SpectralBoundaryEncoder | 241 K  | train | [1, 1, 24136] | [1, 501, 32]                                 
1 | lstm            | LSTM                    | 1.7 M  | train | [1, 501, 32]  | [[1, 501, 256], [[10, 1, 128], [10, 1, 128]]]
2 | fc              | Linear                  | 257    | train | [1, 501, 256] | [1, 501, 1]                                  
3 | loss_fn         | FocalLoss               | 0      | train | ?             | ?                                            
------------------------------------------------------------------------------------------------------------------------------------
2.0 M     Trainable params
1.2 K     Non-trainable param

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

C:\Users\gee8w\AppData\Local\pypoetry\Cache\virtualenvs\sslunsupdet-MT8Ki2i5-py3.11\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


C:\Users\gee8w\AppData\Local\pypoetry\Cache\virtualenvs\sslunsupdet-MT8Ki2i5-py3.11\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 0:  36%|███▌      | 18/50 [00:12<00:22,  1.45it/s, v_num=14, train_loss_step=0.00389]

# Load best ckpt

In [31]:
import numpy as np

best_model = LitEmbeddingLSTMModel.load_from_checkpoint(
    # checkpoint_callback.best_model_path,
    "lstm-detector/lstm_model/version_11/checkpoints/best-lstm-detector.ckpt",
    embedding_model=embedding_model,
)

best_model.eval()

best_model.to("cuda")

LitEmbeddingLSTMModel(
  (embedding_model): SpectralBoundaryEncoder(
    (preprocess): HighPassFilter()
    (transform): ConvTransform(
      (transform): Sequential(
        (0): Conv1d(1, 128, kernel_size=(8,), stride=(4,))
        (1): LeakyReLU(negative_slope=0.01)
        (2): Conv1d(128, 128, kernel_size=(6,), stride=(3,))
        (3): LeakyReLU(negative_slope=0.01)
        (4): Conv1d(128, 128, kernel_size=(4,), stride=(2,))
        (5): LeakyReLU(negative_slope=0.01)
        (6): Conv1d(128, 128, kernel_size=(4,), stride=(2,))
        (7): LeakyReLU(negative_slope=0.01)
      )
    )
    (postprocess): Sequential(
      (0): MLP(
        (net): Sequential(
          (0): Dropout(p=0, inplace=False)
          (1): Linear(in_features=128, out_features=64, bias=True)
          (2): LeakyReLU(negative_slope=0.01)
          (3): Dropout(p=0, inplace=False)
          (4): Linear(in_features=64, out_features=32, bias=True)
        )
      )
      (1): Tanh()
    )
  )
  (lstm): LSTM(3

## Save embeddings

In [ ]:
# trainer = pl.Trainer(accelerator=device,
#     devices=1,)
# outs = trainer.predict(best_model, dataloaders=val_loader)

In [ ]:
# import numpy as np
# import torch
#
# # 1. Pull everything off the GPU and into NumPy
# all_feats = []
# all_labels = []
#
# for batch in outs:
#     # batch["feats"] should be shape [B*T, F] or [B, T, F]
#     feats = batch["feats"].cpu().numpy()
#     labs = batch["labels"].cpu().numpy()
#
#     # if you still have [B, T, F], flatten the time dimension:
#     if feats.ndim == 3:
#         B, T, F = feats.shape
#         feats = feats.reshape(B * T, F)
#         labs  = labs.reshape(B * T)
#
#     all_feats.append(feats)
#     all_labels.append(labs)
#
# all_feats  = np.vstack(all_feats)   # shape [N_total, F]
# all_labels = np.hstack(all_labels)  # shape [N_total]
#
# # 2. Save to a single .npz file
# np.savez_compressed(
#     "lstm_embeddings.npz",
#     embeddings=all_feats,
#     labels=all_labels
# )
# print(f"Saved {all_feats.shape[0]} vectors of dim {all_feats.shape[1]}.")

In [ ]:
# import numpy as np
#
# data = np.load("lstm_embeddings.npz")
# X, y = data["embeddings"], data["labels"]
# print(X.shape, y.shape)  # e.g. (4544, 128), (4544,)

# Save labels and predictions for frontend visualization

In [32]:
# def predict_for_file(file_path):
#     x, _ = librosa.load(
#         file_path,
#         sr=TrainConfig.sample_rate,
#     )
#     x = torch.tensor(x, dtype=torch.float32).unsqueeze(dim=0).unsqueeze(dim=0).to(device)
#     with torch.no_grad():
#         y = best_model(x)
#         probs = torch.sigmoid(y)
#         predictions = (probs > 0.5).nonzero()
#         seconds = predictions.cpu().numpy()[:, 1].reshape(-1) / 1000
#     return seconds.tolist()
#
# predict_for_file("../../data/wavs48khz_watkins/19620917a.wav")

[0.193, 0.374, 0.508, 0.692, 0.695, 0.951, 1.111, 1.17]

In [34]:
# def get_clicks_for_file(file_path):
#     selection_file = f"{TrainConfig.watkins_selections_path[:-5]}{file_path.split('/')[-1][:-4]}.selections.txt"
#     if not os.path.exists(selection_file):
#         selection_file = f"{TrainConfig.watkins_selections_path[:-5]}{file_path.split('/')[-1][:-4]}.q.selections.txt"
#     if os.path.exists(selection_file):
#         # Read the selection file.
#         df = pd.read_csv(selection_file, sep="\t")
#         # Assuming every other row corresponds to a unique selection.
#         df = df.iloc[::2].reset_index(drop=True)
#         begin_times = df["Begin Time (s)"].to_numpy().tolist()
#     print(file_path)
#     return begin_times
#
# get_clicks_for_file("../../data/wavs48khz_watkins/19620917a.wav")

../../data/wavs48khz_watkins/19620917a.wav


[0.19343298,
 0.374481228,
 0.508014843,
 0.695188619,
 0.771112018,
 0.950991725,
 1.111232064,
 1.169986854]

In [35]:
# import pandas as pd
#
# files_to_predict = [
#     "../../data/wavs48khz_watkins/19620917a.wav",
#     "../../data/wavs48khz_watkins/19620917b.wav",
#     "../../data/wavs48khz_watkins/19720806f.wav",
#     "../../data/wavs48khz_watkins/19840322a.wav",
#     "../../data/wavs48khz_watkins/19911021b.wav",
# ]
#
# preds = []
# true_clicks = []
#
# for file_path in files_to_predict:
#     seconds = predict_for_file(file_path)
#     torch.cuda.empty_cache()
#     preds.append({"file_name": file_path.split("/")[-1], "clicks": seconds})
#     clicks = get_clicks_for_file(file_path)
#     true_clicks.append({"file_name": file_path.split("/")[-1], "clicks": clicks})
#
# df = pd.DataFrame(preds)
# df.to_csv("watkins_5_predictions.csv")
# df = pd.DataFrame(true_clicks)
# df.to_csv("watkins_5_trues.csv")

../../data/wavs48khz_watkins/19620917a.wav
../../data/wavs48khz_watkins/19620917b.wav
../../data/wavs48khz_watkins/19720806f.wav
../../data/wavs48khz_watkins/19840322a.wav
../../data/wavs48khz_watkins/19911021b.wav
